# Advanced Analytics + Risk Metrics
**Day 5 | Capstone Project I -- Mutual Fund Analytics**

Covers:
1. Historical VaR (95%) and CVaR -- all 40 schemes
2. Rolling 90-day Sharpe -- 5 key equity funds
3. Investor Cohort Analysis -- by first transaction year
4. SIP Continuity Analysis -- flag at-risk investors
5. Simple Fund Recommender -- by risk appetite
6. Sector HHI Concentration -- equity funds
7. 5 Advanced Insights

**Deliverables:** `var_cvar_report.csv`, `sip_continuity_report.csv`, `recommender.py`, `rolling_sharpe_chart.png`

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)

PROC='../data/processed'; RAW='../data/raw'; CHARTS='../reports/charts'
RF_DAILY=0.065/252; TRADING_DAYS=252
BLUE='#2563EB'; ORANGE='#F97316'; GREEN='#16A34A'
RED='#DC2626';  PURPLE='#7C3AED'; GREY='#6B7280'; TEAL='#0D9488'

fm    = pd.read_csv(f'{PROC}/01_fund_master.csv')
nav   = pd.read_csv(f'{RAW}/02_nav_history.csv', parse_dates=['date'])
txn   = pd.read_csv(f'{PROC}/08_investor_transactions.csv', parse_dates=['transaction_date'])
hld   = pd.read_csv(f'{PROC}/09_portfolio_holdings.csv')
score = pd.read_csv('../fund_scorecard.csv')

nav = nav.sort_values(['amfi_code','date']).reset_index(drop=True)
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
returns_wide = nav.dropna(subset=['daily_return']).pivot(index='date', columns='amfi_code', values='daily_return')
print('Setup complete')

---
## 1. Historical VaR (95%) and CVaR
- **VaR (95%)** = 5th percentile of daily return distribution
- **CVaR (95%)** = mean of returns below the VaR threshold (Expected Shortfall)

In [ ]:
var_rows = []
for code in returns_wide.columns:
    r = returns_wide[code].dropna()
    if len(r) < 30: continue
    var_95  = np.percentile(r, 5)
    cvar_95 = r[r <= var_95].mean()
    var_rows.append({'amfi_code':code, 'var_95_pct':round(var_95*100,4),
                     'cvar_95_pct':round(cvar_95*100,4), 'n_days':len(r)})

var_df = pd.DataFrame(var_rows).merge(
    fm[['amfi_code','fund_house','scheme_name','category','sub_category','plan']], on='amfi_code')
var_df = var_df.sort_values('var_95_pct')
var_df.to_csv('../var_cvar_report.csv', index=False)
print('Top 5 highest-risk (worst VaR):')
print(var_df[['scheme_name','plan','var_95_pct','cvar_95_pct']].head(5).to_string(index=False))
print('\nTop 5 lowest-risk (best VaR):')
print(var_df[['scheme_name','plan','var_95_pct','cvar_95_pct']].tail(5).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,7))
short_v = var_df['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip().str[:25]
bar_c   = [RED if v<-2 else (ORANGE if v<-1.5 else GREEN) for v in var_df['var_95_pct']]
axes[0].barh(short_v, var_df['var_95_pct'], color=bar_c, height=0.65)
axes[0].axvline(-2, color=RED, linestyle='--', linewidth=1, label='VaR=-2%')
axes[0].set_xlabel('95% VaR (%/day)'); axes[0].set_title('Daily VaR (95%) -- All 40 Funds', fontweight='bold')
axes[0].legend(fontsize=9)
colors_cat = {'Equity':BLUE, 'Debt':ORANGE}
for cat, grp in var_df.groupby('category'):
    axes[1].scatter(grp['var_95_pct'], grp['cvar_95_pct'],
                    color=colors_cat.get(cat, GREY), label=cat, s=60, alpha=0.8)
axes[1].set_xlabel('VaR 95% (%/day)'); axes[1].set_ylabel('CVaR 95% (%/day)')
axes[1].set_title('VaR vs CVaR by Category', fontweight='bold'); axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f'{CHARTS}/21_var_cvar_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### Insight 1 -- VaR & CVaR
**Small Cap funds carry the highest daily tail risk: worst VaR of -2.69%/day (SBI Small Cap Direct), meaning on a bad 5% of trading days, investors can lose >2.69% of NAV. Liquid/Debt funds show near-zero VaR (<-0.1%), confirming their capital-preservation role. CVaR consistently runs ~40-60% worse than VaR, reflecting fat left-tail risk in equity funds.**

---
## 2. Rolling 90-Day Sharpe Ratio
`Rolling Sharpe = (rolling_mean(90) - Rf_daily) / rolling_std(90) x sqrt(252)`

In [ ]:
top5 = score[score['category']=='Equity'].head(5)[['amfi_code','scheme_name']].values
colors = [BLUE, ORANGE, GREEN, PURPLE, TEAL]

fig, ax = plt.subplots(figsize=(14,6))
for (code, name), color in zip(top5, colors):
    r = returns_wide[code].dropna()
    roll_sharpe = ((r.rolling(90).mean() - RF_DAILY) / r.rolling(90).std()) * np.sqrt(TRADING_DAYS)
    ax.plot(roll_sharpe.index, roll_sharpe.values,
            label=name.split(' - ')[0].strip(), color=color, linewidth=1.8)

ax.axhline(0,   color=GREY,  linewidth=1,   linestyle='--', alpha=0.7)
ax.axhline(0.5, color=GREEN, linewidth=0.8, linestyle=':',  alpha=0.6, label='Sharpe=0.5')
ax.axvspan(pd.Timestamp('2024-01-01'), pd.Timestamp('2024-06-30'),
           alpha=0.08, color=RED, label='2024 Correction')
ax.set_title('Rolling 90-Day Sharpe Ratio -- Top 5 Equity Funds', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Rolling Sharpe Ratio')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHARTS}/19_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

### Insight 2 -- Rolling Sharpe
**The 2024 correction (Jan-Jun) visibly crushed rolling Sharpe ratios to near-zero or negative for all equity funds -- a regime shift from the 2023 bull run where rolling Sharpes peaked above 2.0. Mirae Asset Large Cap recovered fastest post-correction, suggesting better benchmark alignment and lower volatility drag.**

---
## 3. Investor Cohort Analysis
Group investors by first transaction year. Compare SIP behaviour across cohorts.

In [ ]:
cohort_year = txn.groupby('investor_id')['transaction_date'].min().dt.year.rename('cohort_year')
txn2 = txn.join(cohort_year, on='investor_id')
sip_txn = txn2[txn2['transaction_type']=='SIP']

cohort_sip_avg = sip_txn.groupby('cohort_year')['amount_inr'].mean().round(0).rename('avg_sip_amount')
cohort_total   = txn2[txn2['transaction_type'].isin(['SIP','Lumpsum'])].groupby('cohort_year')['amount_inr'].sum().rename('total_invested_inr')
top_fund = (txn2.groupby(['cohort_year','amfi_code']).size().reset_index(name='count')
            .sort_values('count',ascending=False).drop_duplicates('cohort_year')
            .merge(fm[['amfi_code','scheme_name']], on='amfi_code')[['cohort_year','scheme_name']])

cohort_rpt = cohort_sip_avg.reset_index().merge(cohort_total.reset_index(), on='cohort_year').merge(top_fund, on='cohort_year')
cohort_rpt['total_invested_cr'] = (cohort_rpt['total_invested_inr']/1e7).round(2)
print(cohort_rpt[['cohort_year','avg_sip_amount','total_invested_cr','scheme_name']].to_string(index=False))

### Insight 3 -- Investor Cohorts
**The 2024 cohort (4,803 investors) dominates total invested volume at Rs 225.8 Cr vs the much smaller 2025 cohort (Rs 1.9 Cr, 197 investors). Importantly, 2025 cohort investors invest Rs 2,500 higher average SIP (Rs 13,505 vs Rs 10,997) -- newer investors are entering with larger ticket sizes, reflecting increasing financial awareness. Both cohorts favour growth-oriented equity funds as their top choice.**

---
## 4. SIP Continuity Analysis
Investors with 6+ SIP transactions. Flag those with avg gap > 35 days as 'at-risk'.

In [ ]:
sip_all    = txn[txn['transaction_type']=='SIP'].sort_values(['investor_id','transaction_date'])
sip_counts = sip_all.groupby('investor_id').size()
eligible   = sip_counts[sip_counts>=6].index
sip_elig   = sip_all[sip_all['investor_id'].isin(eligible)].copy()
sip_elig['gap_days'] = sip_elig.groupby('investor_id')['transaction_date'].diff().dt.days
avg_gap = sip_elig.groupby('investor_id')['gap_days'].mean().round(1).rename('avg_gap_days')
gap_df  = avg_gap.reset_index()
gap_df['at_risk'] = gap_df['avg_gap_days'] > 35

at_risk_pct = gap_df['at_risk'].mean() * 100
print(f'Eligible investors (6+ SIP): {len(gap_df):,}')
print(f'At-risk (gap > 35 days)    : {gap_df["at_risk"].sum():,} ({at_risk_pct:.1f}%)')
print(f'Continuity rate            : {100-at_risk_pct:.1f}%')
print(f'Median avg gap             : {gap_df["avg_gap_days"].median():.1f} days')

fig, ax = plt.subplots(figsize=(10,4))
ax.hist(gap_df['avg_gap_days'], bins=40, color=BLUE, alpha=0.7, edgecolor='none')
ax.axvline(35, color=RED, linewidth=2, linestyle='--', label='35-day threshold')
ax.set_xlabel('Avg Gap Between SIP Transactions (days)'); ax.set_ylabel('Number of Investors')
ax.set_title('SIP Continuity -- Distribution of Avg Transaction Gaps', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

### Insight 4 -- SIP Continuity
**97.8% of investors with 6+ SIP transactions are flagged as at-risk (avg gap > 35 days), with a median gap of 64.7 days. This strongly suggests the dataset captures quarterly or bi-monthly SIP patterns rather than monthly -- the 'at-risk' threshold of 35 days may need recalibration for this investor base. True monthly SIP investors (gap ~30 days) represent only 2.2% of the eligible pool.**

---
## 5. Fund Recommender by Risk Appetite
Top 3 funds by Sharpe ratio within matching risk grade.

In [ ]:
import sys; sys.path.insert(0, '..')
from recommender import recommend

for appetite in ['Low', 'Moderate', 'High']:
    df = recommend(appetite)
    print(f'\n--- Risk Appetite: {appetite} ---')
    print(df[['scheme_name','plan','risk_category','sharpe_ratio','cagr_3yr_pct','expense_ratio_pct']].to_string(index=False))

---
## 6. Sector HHI Concentration
`HHI = sum(weight_i^2)` per fund. Range: 0 (perfectly diversified) to 10000 (single-sector).

In [ ]:
import matplotlib.patches as mpatches

eq_codes = fm[fm['category']=='Equity']['amfi_code'].tolist()
hld_eq   = hld[hld['amfi_code'].isin(eq_codes)].copy()

def hhi(weights):
    w = weights / weights.sum()
    return (w**2).sum() * 10000

hhi_rows = []
for code, grp in hld_eq.groupby('amfi_code'):
    sec_wt = grp.groupby('sector')['weight_pct'].sum()
    hhi_rows.append({'amfi_code':code, 'hhi_score':round(hhi(sec_wt),1),
                     'top_sector':sec_wt.idxmax(), 'n_sectors':sec_wt.nunique()})

hhi_df = pd.DataFrame(hhi_rows).merge(fm[['amfi_code','scheme_name','sub_category']], on='amfi_code').sort_values('hhi_score', ascending=False)
print(f'HHI range: {hhi_df["hhi_score"].min():.0f} to {hhi_df["hhi_score"].max():.0f}')
print(f'Most concentrated: {hhi_df.iloc[0]["scheme_name"][:50]}  HHI={hhi_df.iloc[0]["hhi_score"]:.0f}')
print(f'Most diversified : {hhi_df.iloc[-1]["scheme_name"][:50]}  HHI={hhi_df.iloc[-1]["hhi_score"]:.0f}')

In [ ]:
fig2, ax2 = plt.subplots(figsize=(13,7))
short = hhi_df['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip().str[:28]
bar_colors = [RED if h>2500 else (ORANGE if h>1800 else GREEN) for h in hhi_df['hhi_score']]
ax2.barh(short[::-1], hhi_df['hhi_score'][::-1], color=bar_colors[::-1], height=0.7)
ax2.axvline(2500, color=RED,    linestyle='--', linewidth=1, label='HHI=2500 concentrated')
ax2.axvline(1800, color=ORANGE, linestyle=':',  linewidth=1, label='HHI=1800 moderate')
ax2.set_xlabel('HHI Score'); ax2.set_title('Sector HHI Concentration -- Equity Funds', fontweight='bold')
for i,(h,n) in enumerate(zip(hhi_df['hhi_score'][::-1], short[::-1])):
    ax2.text(h+20, i, f'{h:.0f}', va='center', fontsize=8)
ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f'{CHARTS}/20_sector_hhi.png', dpi=150, bbox_inches='tight')
plt.show()

### Insight 5 -- Sector HHI
**Axis Bluechip has the highest HHI (2,968) -- dangerously concentrated in a single sector, creating event risk for investors. UTI Mid Cap is the most diversified (HHI 1,241) with holdings spread across 9+ sectors. Investors seeking true diversification should prefer low-HHI funds; high-HHI funds may outperform when their dominant sector rallies but carry higher drawdown risk during sector corrections.**

---
## Summary -- 5 Advanced Insights

| # | Insight | Metric |
|---|---|---|
| 1 | Small Cap funds have worst VaR (-2.69%/day); Liquid funds near-zero VaR (-0.02%) | VaR/CVaR |
| 2 | 2024 correction crushed rolling Sharpe to near-zero; Mirae Large Cap recovered fastest | Rolling Sharpe |
| 3 | 2025 cohort investors bring Rs 2,500 higher avg SIP ticket vs 2024 cohort | Cohort Analysis |
| 4 | 97.8% of SIP investors show gaps >35 days -- pattern reflects quarterly/bi-monthly SIPs, not monthly | SIP Continuity |
| 5 | Axis Bluechip most concentrated (HHI 2,968); UTI Mid Cap most diversified (HHI 1,241) | Sector HHI |